In [1]:
import pandas as pd
data = pd.read_csv(r'C:\DATA ANALYSIS AND DATA SCIENCE PROJECTS\Multi Scale Energy Stress Analysis\data\ca-buildings_dataset.csv')
data.columns

Index(['Timestamp', 'Building Type', 'Energy Consumption (kWh)',
       'Temperature (°C)', 'Humidity (%)', 'Occupancy Rate (%)',
       'Lighting Consumption (kWh)', 'HVAC Consumption (kWh)',
       'Energy Price ($/kWh)', 'Carbon Emission Rate (g CO2/kWh)',
       'Power Factor', 'Voltage Levels (V)', 'Reactive Power (kVARh)',
       'Power Outage Indicator', 'Indoor Temperature (°C)',
       'Building Age (years)', 'Equipment Age (years)',
       'Energy Efficiency Rating', 'Building Size (m²)',
       'Window-to-Wall Ratio (%)', 'Insulation Quality Score',
       'Historical Energy Consumption (kWh)', 'Maintenance Status',
       'Demand Response Participation', 'Occupancy Schedule',
       'Local Energy Production (kWh)', 'Grid Stability Score',
       'Solar Irradiance (W/m²)', 'Smart Plug Usage (kWh)',
       'Water Usage (liters)', 'Energy Savings Target (%)',
       'Room-Level Energy Consumption (kWh)',
       'Zonal Heating/Cooling Data (kWh)', 'Electric Vehicle Charging Sta

In [2]:
data.isna().sum()

Timestamp                              0
Building Type                          0
Energy Consumption (kWh)               0
Temperature (°C)                       0
Humidity (%)                           0
Occupancy Rate (%)                     0
Lighting Consumption (kWh)             0
HVAC Consumption (kWh)                 0
Energy Price ($/kWh)                   0
Carbon Emission Rate (g CO2/kWh)       0
Power Factor                           0
Voltage Levels (V)                     0
Reactive Power (kVARh)                 0
Power Outage Indicator                 0
Indoor Temperature (°C)                0
Building Age (years)                   0
Equipment Age (years)                  0
Energy Efficiency Rating               0
Building Size (m²)                     0
Window-to-Wall Ratio (%)               0
Insulation Quality Score               0
Historical Energy Consumption (kWh)    0
Maintenance Status                     0
Demand Response Participation          0
Occupancy Schedu

In [6]:
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data['Timestamp'].isnull().sum()

np.int64(0)

In [8]:
import numpy as np

# Temporal
data['hour'] = data['Timestamp'].dt.hour
data['day_of_week'] = data['Timestamp'].dt.dayofweek
data['month'] = data['Timestamp'].dt.month
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)
data['is_peak_hour'] = data['hour'].between(16, 21).astype(int)

# Cyclical encoding
data['hour_sin'] = np.sin(2*np.pi*data['hour']/24)
data['hour_cos'] = np.cos(2*np.pi*data['hour']/24)
data['month_sin'] = np.sin(2*np.pi*data['month']/12)
data['month_cos'] = np.cos(2*np.pi*data['month']/12)

# Consumption ratios (row-level, perfectly valid)
data['hvac_ratio'] = data['HVAC Consumption (kWh)'] / data['Energy Consumption (kWh)']
data['lighting_ratio'] = data['Lighting Consumption (kWh)'] / data['Energy Consumption (kWh)']
data['energy_per_area'] = data['Energy Consumption (kWh)'] / data['Building Size (m²)']
data['net_consumption'] = data['Energy Consumption (kWh)'] - data['Local Energy Production (kWh)']
data['consumption_per_occupancy'] = data['Energy Consumption (kWh)'] / (data['Occupancy Rate (%)'] + 1e-5)

# Thermal / physical
data['temp_diff'] = data['Temperature (°C)'] - data['Indoor Temperature (°C)']
data['hvac_stress'] = data['HVAC Consumption (kWh)'] * np.abs(data['temp_diff'])
data['solar_utilization'] = data['Local Energy Production (kWh)'] / (data['Solar Irradiance (W/m²)'] + 1e-5)

# Power quality
data['apparent_power'] = data['Energy Consumption (kWh)'] / (data['Power Factor'] + 1e-5)
data['reactive_ratio'] = data['Reactive Power (kVARh)'] / (data['Energy Consumption (kWh)'] + 1e-5)
data['voltage_deviation'] = np.abs(data['Voltage Levels (V)'] - 230)

# Building characteristics
data['age_interaction'] = data['Building Age (years)'] * data['Equipment Age (years)']

In [3]:
data['Building Type'].unique()

array(['Residential', 'Industrial', 'Commercial'], dtype=object)

In [5]:
data['Timestamp'].unique()

array(['2018-01-01 00:00:00', '2018-01-01 01:00:00',
       '2018-01-01 02:00:00', ..., '2023-12-31 22:00:00',
       '2023-12-31 23:00:00', '2024-01-01 00:00:00'], dtype=object)

In [12]:
data.isnull().sum().sort_values(ascending=False).head(20)

Timestamp                           0
Building Type                       0
Energy Consumption (kWh)            0
Temperature (°C)                    0
Humidity (%)                        0
Occupancy Rate (%)                  0
Lighting Consumption (kWh)          0
HVAC Consumption (kWh)              0
Energy Price ($/kWh)                0
Carbon Emission Rate (g CO2/kWh)    0
Power Factor                        0
Voltage Levels (V)                  0
Reactive Power (kVARh)              0
Power Outage Indicator              0
Indoor Temperature (°C)             0
Building Age (years)                0
Equipment Age (years)               0
Energy Efficiency Rating            0
Building Size (m²)                  0
Window-to-Wall Ratio (%)            0
dtype: int64

In [13]:
# Thermal envelope efficiency — how well does the building retain temp
data['thermal_efficiency'] = data['Insulation Quality Score'] * (1 - data['Window-to-Wall Ratio (%)'] / 100)

# Heat index approximation (temp + humidity interaction)
data['heat_index'] = data['Temperature (°C)'] + 0.33 * data['Humidity (%)'] - 4

# Historical deviation — is current consumption above/below historical norm
data['consumption_vs_historical'] = data['Energy Consumption (kWh)'] - data['Historical Energy Consumption (kWh)']
data['consumption_vs_historical_ratio'] = data['Energy Consumption (kWh)'] / (data['Historical Energy Consumption (kWh)'] + 1e-5)

# Smart plug as a share of total — non-HVAC flexible load
data['smart_plug_ratio'] = data['Smart Plug Usage (kWh)'] / (data['Energy Consumption (kWh)'] + 1e-5)

# Carbon intensity of consumption
data['total_carbon'] = data['Energy Consumption (kWh)'] * data['Carbon Emission Rate (g CO2/kWh)']

# Water-energy nexus (useful for identifying industrial patterns)
data['water_energy_ratio'] = data['Water Usage (liters)'] / (data['Energy Consumption (kWh)'] + 1e-5)

In [14]:
data.isnull().sum().sort_values(ascending=False).head(20)

Timestamp                           0
Building Type                       0
Energy Consumption (kWh)            0
Temperature (°C)                    0
Humidity (%)                        0
Occupancy Rate (%)                  0
Lighting Consumption (kWh)          0
HVAC Consumption (kWh)              0
Energy Price ($/kWh)                0
Carbon Emission Rate (g CO2/kWh)    0
Power Factor                        0
Voltage Levels (V)                  0
Reactive Power (kVARh)              0
Power Outage Indicator              0
Indoor Temperature (°C)             0
Building Age (years)                0
Equipment Age (years)               0
Energy Efficiency Rating            0
Building Size (m²)                  0
Window-to-Wall Ratio (%)            0
dtype: int64

In [15]:
data.to_csv("buildings_preprocessed.csv", index=False, encoding="utf-8")